# Day 4 — Feature Engineering & Preprocessing Pipeline
**Real Estate Fraud Detection**

Goal: Add stateless + fold-dependent features, build preprocessing pipeline, freeze train/test splits.

**Prerequisites:**
- Day 2 completed → `data/processed/labeled.parquet` (2,147,656 rows)
- Day 3 completed → `reports/eda_findings.md` exists

## 0. Set Project Root

In [1]:
import os
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'configs').exists():
    project_root = project_root.parent
os.chdir(project_root)
print(f'Working directory: {os.getcwd()}')

Working directory: C:\Users\mehal\Downloads\machinelearning\real_estate_fraud_detection


## 1. Imports & Config

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split

from src.ingestion import load_config
from src.features import FeatureEngineer, STATELESS_FEATURES, FOLD_DEPENDENT_FEATURES
from src.preprocessing import build_preprocessor, get_feature_names, save_preprocessor
from src.text_features import TextPipeline, is_text_enabled

CONFIG_PATH = 'configs/config.yaml'
cfg = load_config(CONFIG_PATH)
print(f'Config: {cfg["project"]["name"]} v{cfg["project"]["version"]}')

2026-05-14 09:28:05,235 - src.ingestion - INFO - Config loaded from configs\config.yaml — project: real_estate_fraud_detection v1.1.0


Config: real_estate_fraud_detection v1.1.0


## 2. Load Labeled Data

In [3]:
labeled_path = Path(cfg['data']['processed_path']) / 'labeled.parquet'
df = pd.read_parquet(labeled_path)

target = cfg['columns']['target']
print(f'Loaded  : {df.shape}')
print(f'Fraud % : {df[target].mean()*100:.2f}%')

Loaded  : (2147656, 19)
Fraud % : 7.82%


## 3. Permanent Stratified Subsample — 300k Rows

> **Kyun subsample?**
> - 2.1M rows pe Optuna tuning = 2.5 hrs, kernel timeout
> - 300k rows pe = ~20 min, same model quality
> - Stratified = fraud rate exactly same rehta hai
> - Interview mein bolna: *"2M dataset se 300k stratified sample liya — tree models is size pe well generalize karte hain"*

In [4]:
SAMPLE_SIZE = 300_000

if len(df) > SAMPLE_SIZE:
    # Stratified sample — fraud aur normal dono ka ratio preserve karo
    df = (
        df.groupby(target, group_keys=False)
        .apply(lambda x: x.sample(frac=SAMPLE_SIZE / len(df), random_state=42))
        .reset_index(drop=True)
    )
    print(f'✅ Stratified subsample complete')
else:
    print(f'✅ Dataset already small — no subsample needed')

print(f'Rows after subsample : {len(df):,}')
print(f'Fraud rate           : {df[target].mean()*100:.2f}%  (should match original ~7.37%)')
print(f'Fraud count          : {df[target].sum():,}')

✅ Stratified subsample complete
Rows after subsample : 300,000
Fraud rate           : 7.82%  (should match original ~7.37%)
Fraud count          : 23,468


C:\Users\mehal\AppData\Local\Temp\ipykernel_14632\3735390538.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=SAMPLE_SIZE / len(df), random_state=42))


## 4. Train/Test Split

> **FREEZE:** Test set ab se sirf final evaluation mein use hoga.

In [5]:
# Drop rule_* and fraud_score columns — labels hain, features nahi
rule_cols = [c for c in df.columns if c.startswith('rule_') or c == 'fraud_score']
df_model  = df.drop(columns=rule_cols)

X = df_model.drop(columns=[target])
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=cfg['model']['test_size'],
    random_state=cfg['model']['random_state'],
    stratify=y,
)

print(f'Train : {X_train.shape} | fraud: {y_train.mean()*100:.2f}%')
print(f'Test  : {X_test.shape}  | fraud: {y_test.mean()*100:.2f}%')
print('✅ Test set frozen')

Train : (240000, 10) | fraud: 7.82%
Test  : (60000, 10)  | fraud: 7.82%
✅ Test set frozen


## 5. Stateless Features

In [6]:
X_train = FeatureEngineer.add_stateless_features(X_train, cfg)
X_test  = FeatureEngineer.add_stateless_features(X_test, cfg)

print(f'Stateless features: {STATELESS_FEATURES}')
print(f'Train shape: {X_train.shape}')

2026-05-14 09:28:09,454 - src.features - INFO - Stateless features added — 9 new columns
2026-05-14 09:28:09,594 - src.features - INFO - Stateless features added — 9 new columns


Stateless features: ['price_log', 'house_size_log', 'acre_lot_log', 'price_per_sqft', 'bath_per_bed', 'is_large_property', 'days_since_last_sale', 'sold_year', 'sold_month']
Train shape: (240000, 19)


## 6. Fold-Dependent Features

**Critical:** fit() on train only.

In [7]:
feat_eng = FeatureEngineer(cfg)
feat_eng.fit(X_train, y=y_train)   # train only
X_train = feat_eng.transform(X_train)
X_test  = feat_eng.transform(X_test)

print(f'Fold-dependent features: {FOLD_DEPENDENT_FEATURES}')
print(f'Train shape: {X_train.shape}')

2026-05-14 09:28:09,626 - src.features - INFO - FeatureEngineer.fit() — computing fold-dependent stats
2026-05-14 09:28:10,193 - src.features - INFO -   Cities: 1774 | States: 54 | Zip codes: 22307
2026-05-14 09:28:10,346 - src.features - INFO - Fold-dependent features added — 24 total cols
2026-05-14 09:28:10,399 - src.features - INFO - Fold-dependent features added — 24 total cols


Fold-dependent features: ['city_median_price', 'price_vs_city_median', 'state_median_price', 'zip_listing_count', 'city_fraud_rate']
Train shape: (240000, 24)


## 7. Preprocessing Pipeline

In [8]:
preprocessor = build_preprocessor(cfg, include_city_fraud_rate=True)
preprocessor.fit(X_train)   # train only

X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

feature_names = get_feature_names(cfg, include_city_fraud_rate=True)
print(f'Train processed : {X_train_proc.shape}')
print(f'Test  processed : {X_test_proc.shape}')
print(f'Features        : {len(feature_names)}')

# NaN check
train_nans = np.isnan(X_train_proc).sum()
test_nans  = np.isnan(X_test_proc).sum()
print(f'NaNs train: {train_nans} | NaNs test: {test_nans}')
print('✅ No NaNs' if train_nans == 0 and test_nans == 0 else '⚠️ NaNs found')

2026-05-14 09:28:10,414 - src.preprocessing - INFO - Preprocessor built — 19 numerical, 2 categorical


Train processed : (240000, 21)
Test  processed : (60000, 21)
Features        : 21
NaNs train: 0 | NaNs test: 0
✅ No NaNs


## 8. Save Everything

In [9]:
splits_path = Path(cfg['data']['splits_path'])
splits_path.mkdir(parents=True, exist_ok=True)

# Splits
X_train.to_parquet(splits_path / 'X_train.parquet', index=False)
X_test.to_parquet(splits_path  / 'X_test.parquet',  index=False)
y_train.to_frame().to_parquet(splits_path / 'y_train.parquet', index=False)
y_test.to_frame().to_parquet(splits_path  / 'y_test.parquet',  index=False)
print(f'✅ Splits saved → {splits_path}')

# Preprocessor
save_preprocessor(preprocessor, cfg)
print(f'✅ Preprocessor → {cfg["paths"]["preprocessing_pipeline"]}')

# FeatureEngineer
fe_path = Path(cfg['data']['processed_path']) / 'feature_engineer.pkl'
with open(fe_path, 'wb') as f:
    pickle.dump(feat_eng, f)
print(f'✅ FeatureEngineer → {fe_path}')

2026-05-14 09:28:12,967 - src.preprocessing - INFO - Preprocessor saved → models/preprocessing_pipeline.pkl


✅ Splits saved → data\splits
✅ Preprocessor → models/preprocessing_pipeline.pkl
✅ FeatureEngineer → data\processed\feature_engineer.pkl


## 9. Day 4 Exit Criteria

In [10]:
print('=== DAY 4 EXIT CRITERIA ===')

fraud_rate_ok = abs(df[target].mean() - y_train.mean()) < 0.005

checks = [
    (f'Subsample = {len(X_train)+len(X_test):,} rows',
     len(X_train) + len(X_test) <= SAMPLE_SIZE + 2000),
    ('Fraud rate preserved after subsample',
     fraud_rate_ok),
    ('Stateless features added',
     all(c in X_train.columns for c in STATELESS_FEATURES[:3])),
    ('Fold-dependent fit on train only',
     feat_eng._fitted),
    ('No NaNs after preprocessing',
     train_nans == 0 and test_nans == 0),
    ('All 4 split files saved',
     all((splits_path/f).exists() for f in
         ['X_train.parquet','X_test.parquet','y_train.parquet','y_test.parquet'])),
    ('Preprocessor saved',
     Path(cfg['paths']['preprocessing_pipeline']).exists()),
]

all_pass = True
for label, result in checks:
    icon = '☑' if result else '☒'
    if not result: all_pass = False
    print(f'  {icon} {label}')

print(f'\n{"✅ All passed" if all_pass else "⚠️ Some failed"}')
print(f'\nFinal sizes:')
print(f'  X_train : {X_train.shape}  — ~5 min per CV fold now')
print(f'  X_test  : {X_test.shape}')
print(f'\n→ Ready for Day 5 — Baseline Models')
print(f'→ Day 6 Optuna tuning ~20 min (was 2.5 hrs)')

=== DAY 4 EXIT CRITERIA ===
  ☑ Subsample = 300,000 rows
  ☑ Fraud rate preserved after subsample
  ☑ Stateless features added
  ☑ Fold-dependent fit on train only
  ☑ No NaNs after preprocessing
  ☑ All 4 split files saved
  ☑ Preprocessor saved

✅ All passed

Final sizes:
  X_train : (240000, 24)  — ~5 min per CV fold now
  X_test  : (60000, 24)

→ Ready for Day 5 — Baseline Models
→ Day 6 Optuna tuning ~20 min (was 2.5 hrs)
